In [49]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 1000)

In [50]:
df_commercial_activity = pd.read_csv('../data/datasets_TFM + diccionario/customer_commercial_activity.csv', sep=',')
df_sociodemographics = pd.read_csv('../data/datasets_TFM + diccionario/customer_sociodemographics.csv', sep=',')
df_customer_products = pd.read_csv('../data/datasets_TFM + diccionario/customer_products.csv', sep=',')
df_sales = pd.read_csv('../data/datasets_TFM + diccionario/sales.csv', sep=',')

df_product_description = pd.read_csv('../data/datasets_TFM + diccionario/product_description.csv', sep=',')

In [3]:
df_product_description.family_product.unique()

array(['account', 'payment_card', 'pension_plan', 'investment', 'loan'],
      dtype=object)

In [4]:
df_commercial_activity.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_sociodemographics.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_customer_products.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_sales.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')
df_product_description.drop(columns=['Unnamed: 0'], inplace=True, errors='ignore')

In [5]:
df_sales.rename(columns={"cid":"pk_cid"}, inplace = True)
df_sales['pk_cid'] = df_sales['pk_cid'].astype(str)

# 1. Preprocessing + FE

### CUSTOMER DATA

In [6]:
df = df_commercial_activity.merge(df_sociodemographics, how='left', on = ['pk_cid','pk_partition'])
df = df.merge(df_customer_products, how='left', on = ['pk_cid','pk_partition'])
df

,pk_cid,pk_partition,entry_date,entry_channel,active_customer,segment,country_id,region_code,gender,age,deceased,salary,short_term_deposit,loans,mortgage,funds,securities,long_term_deposit,em_account_pp,credit_card,payroll,pension_plan,payroll_account,emc_account,debit_card,em_account_p,em_acount
0,1375586,2018-01,2018-01,KHL,1.0,02 - PARTICULARES,ES,29.0,H,35,N,87218.10,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
1,1050611,2018-01,2015-08,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,23,N,35548.74,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
2,1050612,2018-01,2015-08,KHE,0.0,03 - UNIVERSITARIO,ES,13.0,V,23,N,122179.11,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
3,1050613,2018-01,2015-08,KHD,0.0,03 - UNIVERSITARIO,ES,50.0,H,22,N,119775.54,1,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,0
4,1050614,2018-01,2015-08,KHE,1.0,03 - UNIVERSITARIO,ES,50.0,V,23,N,NaN,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5962919,1166765,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,50.0,V,22,N,43912.17,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
5962920,1166764,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,26.0,V,23,N,23334.99,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
5962921,1166763,2019-05,2016-08,KHE,1.0,02 - PARTICULARES,ES,50.0,H,47,N,NaN,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1
5962922,1166789,2019-05,2016-08,KHE,0.0,03 - UNIVERSITARIO,ES,50.0,H,22,N,199592.82,0,0,0,0,0,0,0,0,0.0,0.0,0,0,0,0,1


In [7]:
print("Número de registros: {}".format(df.pk_cid.count())) #5.962.924
print("Número de ids únicos: {}".format(df.pk_cid.nunique())) #456.373
print("Número de particiones únicos: {}".format(df.pk_partition.nunique())) #17

Número de registros: 5962924
Número de ids únicos: 456373
Número de particiones únicos: 17


In [8]:
df["pk_cid"] = df["pk_cid"].astype(str)

#### -> Imputación de nulos

In [9]:
df.select_dtypes(include=['number']).isnull().sum().sort_values(ascending=False)

salary                1541104
region_code              2264
pension_plan               61
payroll                    61
active_customer             0
credit_card                 0
em_account_p                0
debit_card                  0
emc_account                 0
payroll_account             0
long_term_deposit           0
em_account_pp               0
securities                  0
funds                       0
mortgage                    0
loans                       0
short_term_deposit          0
age                         0
em_acount                   0
dtype: int64

In [10]:
df.select_dtypes(include=['object']).isnull().sum().sort_values(ascending=False)

segment          133944
entry_channel    133033
gender               25
pk_cid                0
pk_partition          0
entry_date            0
country_id            0
deceased              0
dtype: int64

In [11]:
cols_with_nulls = ['salary','region_code']

for col in cols_with_nulls:
    null_pct = (df[col].isnull().sum() / len(df)) * 100
    print(f'% de nulos en {col}: {null_pct:.2f}%')

% de nulos en salary: 25.84%
% de nulos en region_code: 0.04%


In [12]:
# 1. Variables numericas

# 1.1 salario
median_salary_cid = df.groupby('pk_cid')['salary'].median()
median_salary_regioncode = df.groupby('region_code')['salary'].median()

df.loc[(df["age"]<18)&(df["salary"].isna()), 'salary'] = 0 # vemos que todos los menores tienen el salary null, lo imputamos con 0
df['salary'] = df['salary'].fillna(df['pk_cid'].map(median_salary_cid))
df['salary'] = df['salary'].fillna(df['region_code'].map(median_salary_regioncode))
df['salary'] = df['salary'].fillna(df['salary'].median())  # fallback global

# 1.2 region_code (no es realmente numerica)
mode_region_cid = df.groupby('pk_cid')['region_code'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
mode_region_es = df[df['country_id'] == 'ES'].groupby('pk_partition')['region_code'].agg(lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)

def fill_region(row):
    if pd.notna(row['region_code']):
        return row['region_code']
    
    cid_mode = mode_region_cid.get(row['pk_cid'], np.nan)
    if pd.notna(cid_mode):
        return cid_mode
    
    if row['country_id'] == 'ES':
        return mode_region_es.get(row['pk_partition'], np.nan)
    else:
        return -1

df['region_code'] = df.apply(fill_region, axis=1)


# 1.3 para todo producto que no tenga valor, lo imputamos como 0
product_cols = ['short_term_deposit', 'loans', 'mortgage',
       'funds', 'securities', 'long_term_deposit', 'em_account_pp',
       'credit_card', 'payroll', 'pension_plan', 'payroll_account',
       'emc_account', 'debit_card', 'em_account_p', 'em_acount']

for col in product_cols:
    df[col] = df[col].fillna(0)


In [13]:
# 2. Variables categóricas

# 2.1 rellenamos los pocos nulos que hay con la moda de su partición
mode_segment = df.groupby('pk_partition')['segment'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
mode_channel = df.groupby('pk_partition')['entry_channel'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

df['segment'] = df['segment'].fillna(df['pk_partition'].map(mode_segment))
df['entry_channel'] = df['entry_channel'].fillna(df['pk_partition'].map(mode_channel)) 

# 2.2 geneder
mode_gender_cid = df.groupby('pk_cid')['gender'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
mode_gender_partition = df.groupby('pk_partition')['gender'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)

df['gender'] = df['gender'].fillna(df['pk_cid'].map(mode_gender_cid))
df['gender'] = df['gender'].fillna(df['pk_partition'].map(mode_gender_partition))
df['gender'] = df['gender'].fillna(df['gender'].mode()[0])  # fallback global


In [14]:
df.select_dtypes(include=['number']).isnull().sum().sort_values(ascending=False)

active_customer       0
em_account_pp         0
em_account_p          0
debit_card            0
emc_account           0
payroll_account       0
pension_plan          0
payroll               0
credit_card           0
long_term_deposit     0
region_code           0
securities            0
funds                 0
mortgage              0
loans                 0
short_term_deposit    0
salary                0
age                   0
em_acount             0
dtype: int64

In [15]:
df.select_dtypes(include=['object']).isnull().sum().sort_values(ascending=False)

pk_cid           0
pk_partition     0
entry_date       0
entry_channel    0
segment          0
country_id       0
gender           0
deceased         0
dtype: int64

In [16]:
print("Número de registros: {}".format(df.pk_cid.count())) #5.962.924
print("Número de ids únicos: {}".format(df.pk_cid.nunique())) #456.373
print("Número de particiones únicos: {}".format(df.pk_partition.nunique())) #17

Número de registros: 5962924
Número de ids únicos: 456373
Número de particiones únicos: 17


#### dtypes transformation

In [17]:
df['pk_partition'] = pd.to_datetime(df['pk_partition'], errors='coerce')
df['entry_date'] = pd.to_datetime(df['entry_date'], errors='coerce')

## New interesting features

* z_months_since_entry

In [18]:
df['z_months_since_entry'] = ((df['pk_partition'] - df['entry_date']).dt.days / 30).round()

In [19]:
df.drop(columns=['entry_date'], inplace=True)

* z_pct_months_active

In [20]:
df = df.sort_values(['pk_cid', 'pk_partition'])

df['active_flag'] = df['active_customer'].eq(1).astype(int)
df['cum_months_total'] = df.groupby('pk_cid').cumcount() + 1
df['cum_months_active'] = df.groupby('pk_cid')['active_flag'].cumsum()

# Calculamos el porcentaje incremental
df['z_pct_months_active'] = (df['cum_months_active'] / df['cum_months_total']).round(2)
df = df.drop(columns=['active_flag', 'cum_months_total', 'cum_months_active'])

* z_num_products

In [21]:
df['z_num_products'] = df[product_cols].sum(axis=1)

* z_num_families

In [27]:
df.head()

,pk_cid,pk_partition,entry_channel,active_customer,segment,country_id,region_code,gender,age,deceased,salary,short_term_deposit,loans,mortgage,funds,securities,long_term_deposit,em_account_pp,credit_card,payroll,payroll_account,emc_account,debit_card,em_account_p,em_acount,z_months_since_entry,z_pct_months_active,z_num_products,account,pension_plan,payment_card,loan,investment,z_num_families
83145,1000028,2018-01-01,KHC,1.0,02 - PARTICULARES,ES,28.0,H,43,N,133378.89,0,0,0,0,0,0,0,0,0.0,0,0,1,0,1,12.0,1.0,2.0,1,0,1,0,0,2
398328,1000028,2018-02-01,KHC,1.0,02 - PARTICULARES,ES,28.0,H,43,N,133378.89,0,0,0,0,0,0,0,0,0.0,0,0,1,0,1,13.0,1.0,2.0,1,0,1,0,0,2
648478,1000028,2018-03-01,KHC,1.0,02 - PARTICULARES,ES,28.0,H,43,N,133378.89,0,0,0,0,0,0,0,0,0.0,0,0,1,0,1,14.0,1.0,2.0,1,0,1,0,0,2
895292,1000028,2018-04-01,KHC,1.0,02 - PARTICULARES,ES,28.0,H,43,N,133378.89,0,0,0,0,0,0,0,0,0.0,0,0,1,0,1,15.0,1.0,2.0,1,0,1,0,0,2
1055025,1000028,2018-05-01,KHC,1.0,02 - PARTICULARES,ES,28.0,H,43,N,133378.89,0,0,0,0,0,0,0,0,0.0,0,0,1,0,1,16.0,1.0,2.0,1,0,1,0,0,2


In [ ]:
product_to_family = df_product_description.set_index('product_desc')['family_product'].to_dict()

product_family_map = {
    col: product_to_family[col]
    for col in product_cols
    if col in product_to_family
}

df_families = pd.DataFrame({
    family: (df[[col for col, fam in product_family_map.items() if fam == family]].max(axis=1) > 0).astype(int)
    for family in set(product_family_map.values())
})

df.drop(columns='pension_plan', inplace=True)
df = pd.concat([df, df_families], axis=1)
df['z_num_families'] = df_families.sum(axis=1)

* z_active_with_product

In [30]:
product_cols_wo_pension_plan = ['short_term_deposit',
 'loans',
 'mortgage',
 'funds',
 'securities',
 'long_term_deposit',
 'em_account_pp',
 'credit_card',
 'payroll',
 'payroll_account',
 'emc_account',
 'debit_card',
 'em_account_p',
 'em_acount']

In [31]:
# Creamos nueva variable de actividad 
# Si el cliente no tiene ningún producto realmente no es cliente activo
df['z_active_with_product'] = (~(df[product_cols] == 0).all(axis=1)).astype(int)

In [32]:
print("Número de registros: {}".format(df.pk_cid.count())) #5.962.924
print("Número de ids únicos: {}".format(df.pk_cid.nunique())) #456.373
print("Número de particiones únicos: {}".format(df.pk_partition.nunique())) #17

Número de registros: 5962924
Número de ids únicos: 456373
Número de particiones únicos: 17


In [33]:
df.drop(columns=product_cols_wo_pension_plan, inplace=True)

* geographic features

In [34]:
df['region_code'] = df['region_code'].astype(int)

macro_region_map = {
    # NORTE (Galicia, Asturias, Cantabria, País Vasco, Navarra, La Rioja)
    15: 'norte', 27: 'norte', 32: 'norte', 33: 'norte', 39: 'norte', 
    48: 'norte', 20: 'norte', 31: 'norte',

    # NOROESTE / CASTILLA Y LEÓN
    24: 'noroeste', 34: 'noroeste', 37: 'noroeste', 9: 'noroeste',
    47: 'noroeste', 42: 'noroeste', 40: 'noroeste', 5: 'noroeste', 49: 'noroeste',

    # CENTRO (Madrid + Castilla-La Mancha + Extremadura)
    28: 'centro', 
    45: 'centro', 13: 'centro', 16: 'centro', 19: 'centro', 45: 'centro', 2: 'centro', 
    10: 'centro', 6: 'centro',

    # MEDITERRÁNEO (Cataluña + C. Valenciana + Murcia + Baleares)
    8: 'mediterraneo', 17: 'mediterraneo', 25: 'mediterraneo', 43: 'mediterraneo', # Cataluña
    3: 'mediterraneo', 12: 'mediterraneo', 46: 'mediterraneo', # C. Valenciana
    30: 'mediterraneo', # Murcia
    7: 'mediterraneo',  # Baleares

    # SUR (Andalucía)
    4: 'sur', 11: 'sur', 14: 'sur', 18: 'sur', 21: 'sur', 
    23: 'sur', 29: 'sur', 41: 'sur',

    # CANARIAS
    35: 'islas', 38: 'islas',

    # CEUTA & MELILLA
    51: 'islas', 52: 'islas',

    # EXTRANJEROS
    -1: 'extranjero'
}

df['macro_region'] = df['region_code'].map(macro_region_map).fillna('otros')

In [35]:
provincias_costa = {
    15, 27, 32, 33, 39, 48, 20,                      # norte costa
    3, 12, 46, 30, 7,                                 # mediterráneo
    8, 17, 25, 43,                                     # cataluña
    4, 11, 14, 18, 21, 23, 29, 41,                     # sur costa
    35, 38, 51, 52                                     # islas
}

df['is_coast'] = df['region_code'].apply(lambda x: 1 if x in provincias_costa else 0)

In [36]:
grandes_ciudades = {28, 8, 46, 11, 41, 29, 30, 3, 33}  # Madrid, Barcelona, Valencia, Sevilla, Málaga, Murcia, Alicante, Gijón
ciudades_medianas = {12, 7, 4, 25, 14, 24, 21, 15, 27, 17, 43, 48, 20, 35, 38}
# El resto: localidad pequeña o rural

def clasificar_tamano(rc):
    if rc == -1:
        return 'extranjero'
    elif rc in grandes_ciudades:
        return 'gran_ciudad'
    elif rc in ciudades_medianas:
        return 'ciudad_mediana'
    else:
        return 'pueblo'

df['city_size'] = df['region_code'].apply(clasificar_tamano)

In [37]:
df['national'] = df['country_id'].apply(lambda x: 0 if x != "ES" else 1)

In [38]:
df.drop(columns=['region_code', 'country_id'], inplace=True)

* net_margin, avg_margin_per_month

In [39]:
# # no lo hacemos por partición porque luego nos da igual, cogeremos la última.
# df_margin = df_sales.groupby("pk_cid")["net_margin"].sum().reset_index()
# df_margin.rename(columns={"net_margin": "total_margin"}, inplace=True)

# df = df.merge(df_margin, on='pk_cid', how='left')


# df['avg_margin_per_month'] = df.apply(
#     lambda row: row['total_margin'] / row['z_months_since_entry'] if row['z_months_since_entry'] > 0 
#                 else 0,
#     axis=1
# ).fillna(0)

# df['total_margin'] = df['total_margin'].fillna(0)
# df['avg_margin_per_month'] = df['avg_margin_per_month'].round(2)

* months_since_last_purchase

In [40]:
# # no lo hacemos por partición porque luego nos da igual, cogeremos la última.
# df_sales["month_sale"] = pd.to_datetime(df_sales["month_sale"])
# df_last_purchase = df_sales.groupby("pk_cid")["month_sale"].max().reset_index().rename(columns={"month_sale": "last_purchase_date"})
# df = df.merge(df_last_purchase, on ='pk_cid', how='left')

In [41]:
# df['months_since_last_purchase'] = (
#     (df['pk_partition'].dt.year - df['last_purchase_date'].dt.year) * 12 +
#     (df['pk_partition'].dt.month - df['last_purchase_date'].dt.month)
# )

In [42]:
# para los que nunca han contratado producto rellenar con la fecha de antiguedad máxima del dataset
# df['months_since_last_purchase'] = df['months_since_last_purchase'].fillna(999)

In [43]:
# df.drop(columns={'last_purchase_date'}, inplace=True)

# 2. Data Preparation for future Modelling

* Para la segmentación nos interesa quedarnos con la última foto de la cartera.

In [44]:
df_model = df[df["pk_partition"]==df["pk_partition"].max()]

In [45]:
df_model.head()

,pk_cid,pk_partition,entry_channel,active_customer,segment,gender,age,deceased,salary,z_months_since_entry,z_pct_months_active,z_num_products,account,pension_plan,payment_card,loan,investment,z_num_families,z_active_with_product,macro_region,is_coast,city_size,national
5816883,1000028,2019-05-01,KHC,1.0,02 - PARTICULARES,H,44,N,133378.89,28.0,1.00,2.0,1,0,1,0,0,2,1,centro,0,gran_ciudad,1
5816882,1000096,2019-05-01,KFA,1.0,02 - PARTICULARES,H,10,N,0.00,53.0,1.00,0.0,0,0,0,0,0,0,0,centro,0,gran_ciudad,1
5816887,1000113,2019-05-01,KAT,0.0,02 - PARTICULARES,V,54,N,136705.50,50.0,0.00,0.0,0,0,0,0,0,0,0,sur,1,ciudad_mediana,1
5816886,1000130,2019-05-01,KAT,1.0,02 - PARTICULARES,H,44,N,126133.50,14.0,0.27,0.0,0,0,0,0,0,0,0,centro,0,gran_ciudad,1
5816889,1000157,2019-05-01,KFC,1.0,02 - PARTICULARES,V,44,N,68038.20,39.0,0.94,1.0,1,0,0,0,0,1,1,mediterraneo,1,gran_ciudad,1


In [46]:
# df_model[df_model["total_margin"]==0].pk_cid.nunique()/df_model.pk_cid.nunique()

In [47]:
print(len(df), len(df_model))
print(df.pk_cid.nunique(), df_model.pk_cid.nunique())

5962924 442995
456373 442995


In [48]:
# apartir de aquí, hacer transformaciones de variables (ej. encodings) y luego clusterizar
df_model.to_csv('data/df_for_clustering.csv')